# 📊 sm75 ガード全一覧

wheel 内の capability 分岐を抽出して T4 の可否を表にする。


In [ ]:
import glob
import pathlib
import subprocess
import sys
import zipfile

subprocess.run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
                "--no-deps", "vllm==0.32.0", "-d", "/tmp/w"],
               capture_output=True, text=True)
whl = glob.glob("/tmp/w/*.whl")
print("wheel:", whl)
if not whl:
    raise SystemExit("wheel 取得失敗")
zipfile.ZipFile(whl[0]).extractall("/tmp/vl")
root = pathlib.Path("/tmp/vl/vllm")
print("py files:", len(list(root.rglob("*.py"))))

import collections
import re

DIRS = ("attention", "quantization", "model_executor/layers", "platforms",
        "v1/attention", "v1/spec_decode", "distributed", "compilation")
pat = re.compile("capability|is_sm|sm75|sm80|sm90")

rows = []
for f in sorted(root.rglob("*.py")):
    rel = str(f.relative_to(root))
    if not any(rel.startswith(d) for d in DIRS):
        continue
    try:
        t = f.read_text(errors="replace")
    except Exception:
        continue
    for i, l in enumerate(t.splitlines(), 1):
        s = l.strip()
        if not pat.search(s):
            continue
        if not any(k in s for k in ("<", ">", "==", "if ", "elif ", "raise", "assert")):
            continue
        rows.append((rel, i, s[:140]))

print("hit count:", len(rows))
by_file = collections.Counter(r[0] for r in rows)
print("\n=== ファイル別件数（上位 20）===")
for name, n in by_file.most_common(20):
    print(f"  {n:4d}  {name}")

print("\n=== < 80 / < 75 / == 75 の行（T4 で分岐する箇所）===")
for rel, i, s in rows:
    if re.search(r"[<>]=?\s*(75|80|89|90)|==\s*(75|80|90)", s):
        print(f"  {rel}:{i}: {s}")

print("\n=== marlin / fp8 / flashinfer の可否 ===")
KEYS = ["marlin", "fp8", "flashinfer", "xformers", "triton"]
for k in KEYS:
    n = sum(1 for r in rows if k in r[2].lower())
    print(f"  {k}: {n} 行で capability 判定と同居")
